# 📝 การบ้าน Day 1: Data Engineering Pipeline
## Agentic RAG: From Zero to Hero

---

### 📋 คำชี้แจง

1. **ให้ทำด้วยตนเอง** — ห้ามใช้ AI ช่วยเขียนโค้ด
2. **ห้ามลอกกัน** — ข้อมูลของแต่ละคนจะ **ไม่เหมือนกัน** (สร้างจากรหัสนักศึกษา)
3. **ส่ง notebook นี้** พร้อมผลลัพธ์ที่ run แล้ว (.ipynb)
4. **คะแนน**: 10 คะแนน

> ⚠️ **ระบบจะตรวจจับการลอก** จากค่า hash, embedding, และ score ที่ต้องตรงกับรหัสนักศึกษาของแต่ละคน

## 📦 Dependencies

In [ ]:
%%time
!pip install -q sentence-transformers qdrant-client langchain-text-splitters rank_bm25 pythainlp scikit-learn

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ !')

## 🎓 Enter student information

In [ ]:
# ─── ───
STUDENT_NAME = '' # ' '
STUDENT_ID = '' # '6512345678'
PHONE = '' # '081-234-5678'
LINE_ID = '' # 'somchai.j'

# ─── () ───
assert len(STUDENT_ID) >= 5, '❌ !'
assert len(STUDENT_NAME) >= 2, '❌ -!'

print(f'✅ -: {STUDENT_NAME}')
print(f'✅ : {STUDENT_ID}')
print(f'📱 : {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 Step 2: Create a personalized dataset (do not edit this cell)

In [ ]:
%%time
# ===== Do not edit this cell =====
# Create a student-specific dataset from the student ID

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

# เนื้อหาหลัก — สลับลำดับและเลือกเนื้อหาตามรหัสนักศึกษา
all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ โดยไม่จำเป็นต้องเขียนโปรแกรมอย่างชัดเจนสำหรับทุกกรณี',
    'Deep Learning เป็นเทคนิคหนึ่งของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา และการสร้างข้อความ',
    'Natural Language Processing หรือ NLP คือสาขาที่เกี่ยวข้องกับการทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์ การสรุปข้อความ และการตอบคำถาม',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้อง ทันสมัย และอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้สามารถค้นหาข้อมูลที่มีความหมายคล้ายคลึงกันได้อย่างรวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลขหรือ Vector ที่สามารถแสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้คอมพิวเตอร์สามารถเปรียบเทียบความคล้ายคลึงระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูลแบบลำดับ เป็นพื้นฐานของโมเดลภาษาขนาดใหญ่อย่าง GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่งหรือ Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพและความแม่นยำของคำตอบอย่างมาก',
    'Fine-tuning คือกระบวนการปรับแต่งโมเดลที่ผ่านการฝึกมาแล้วด้วยข้อมูลเฉพาะทางเพิ่มเติม เพื่อให้โมเดลทำงานได้ดีขึ้นในงานที่เฉพาะเจาะจง เช่น การวิเคราะห์เอกสารทางการแพทย์',
    'Tokenization คือกระบวนการแบ่งข้อความออกเป็นหน่วยย่อยที่เรียกว่า Token ซึ่งอาจเป็นคำ คำย่อย หรือตัวอักษร สำหรับภาษาไทยการตัดคำมีความซับซ้อนเพราะไม่มีเว้นวรรคระหว่างคำ',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic Chunking',
    'Cosine Similarity เป็นวิธีวัดความคล้ายคลึงระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน ค่า 0 หมายถึงตั้งฉาก นิยมใช้ในงาน NLP และ Information Retrieval',
]

# สลับลำดับตาม seed ของนักศึกษา
random.shuffle(all_paragraphs)

# เลือก 8 ย่อหน้า + สร้าง duplicate 1 ตัว
selected = all_paragraphs[:8]
duplicate_idx = random.randint(0, 5)
selected.append(selected[duplicate_idx])  # ย่อหน้าที่ซ้ำ

# บันทึกเป็นไฟล์
os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID} เรียบร้อย!')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์ (มี 1 ไฟล์ที่ซ้ำ)')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

---
## 🎯 โจทย์: สร้าง Data Engineering Pipeline

ให้สร้าง **RAG Data Pipeline** จากชุดข้อมูลเฉพาะตัวของคุณ ทำตามขั้นตอนด้านล่าง
แต่ละขั้นตอนมี **คำตอบที่ต้องรายงาน** — ค่าเหล่านี้จะไม่ซ้ำกันระหว่างนักศึกษา

---

### ขั้นตอนที่ 1: ตรวจหา Duplicate ด้วย MD5 (2 คะแนน)

- คำนวณ MD5 hash ของทุกไฟล์ใน `homework_data/`
- ค้นหาไฟล์ที่ซ้ำกัน
- ลบไฟล์ที่ซ้ำออก (เก็บตัวแรก)

**📝 รายงาน:**
1. ไฟล์คู่ไหนที่ซ้ำกัน?
2. MD5 hash ของไฟล์ที่ซ้ำคือเท่าไร?
3. หลังลบแล้วเหลือกี่ไฟล์?

In [ ]:
# 1: duplicate 

# 💡 Hint:
# 1. import hashlib
# 2. compute_md5(filepath) return hash
# 3. 'homework_data/' hash dict
# 4. hash → duplicate



# ✅ Self-check (run )
# assert dup_hash is not None, '❌ duplicate'
# assert files_remaining >= 7, '❌ 7'
# print(f'✅ Step 1 passed: ={dup_files}, MD5={dup_hash}')

### 2: Chunking (2 )

- ( duplicate)
- Chunk `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`

**📝 :**
1. chunks?
2. Chunk 1 ? (copy )
3. Chunk ?

In [ ]:
# 2: chunking 

# 💡 Hint:
# 1. from langchain_text_splitters import RecursiveCharacterTextSplitter
# 2. 
# 3. splitter chunk_size=150, chunk_overlap=30
# 4. splitter.split_text(all_text)



# ✅ Self-check
# assert len(chunks) > 0, '❌ chunk'
# assert len(chunks[0]) <= 150, '❌ chunk '
# print(f'✅ Step 2 passed: {len(chunks)} chunks, chunk 1 = {len(chunks[0])} chars')

### 3: Embedding + Similarity (3 )

- embedding `intfloat/multilingual-e5-large`
- **Cosine Similarity** query chunk
- Query: `'query: '`

**📝 :**
1. Chunk similarity ? ( chunk score 4 )
2. Chunk similarity ? ( chunk score 4 )
3. — chunk ? ( 2-3 )

In [ ]:
# 3: embedding + similarity 
# prefix 'query: ' 'passage: ' 

# 💡 Hint:
# 1. from sentence_transformers import SentenceTransformer
# 2. model = SentenceTransformer('intfloat/multilingual-e5-large')
# 3. query = 'query: '
# 4. passages = ['passage: ' + c for c in chunks]
# 5. query_emb = model.encode(query)
# 6. passage_embs = model.encode(passages)
# 7. cosine_similarity() sklearn



# ✅ Self-check
# assert best_score > 0.5, '❌ similarity score prefix'
# print(f'✅ Step 3 passed: best chunk={best_idx}, score={best_score:.4f}')

### ขั้นตอนที่ 4: Qdrant Retrieval (3 คะแนน)

- สร้าง Qdrant client (in-memory) + Collection ชื่อ `f'hw_{STUDENT_ID}'`
- Upsert ทุก chunk + embedding เข้า Qdrant
- ค้นหาด้วย query: `'query: การแบ่งข้อความออกเป็นส่วนย่อย'`
- ใช้ `top_k=3`

**📝 รายงาน:**
1. ผลลัพธ์ Top-3 แต่ละอันดับ มี score เท่าไร? (ทศนิยม 4 ตำแหน่ง)
2. อันดับ 1 เป็นเนื้อหาเกี่ยวกับอะไร?
3. เขียนสรุป: ถ้าจะนำระบบนี้ไปใช้งานจริง ต้องปรับปรุงอะไรบ้าง? (อธิบาย 3-5 ประโยค)

In [ ]:
# 4: Qdrant retrieval 

# 💡 Hint:
# 1. from qdrant_client import QdrantClient, models
# 2. qdrant = QdrantClient(':memory:')
# 3. collection f'hw_{STUDENT_ID}'
# 4. Upsert chunk + embedding Qdrant
# 5. query = 'query: '
# 6. qdrant.query_points(..., limit=3)



# ✅ Self-check
# assert len(results) == 3, '❌ top_k=3 '
# print(f'✅ Step 4 passed: Top-3 scores={[r.score for r in results]}')

## 📊 

| | | |
|---------|:-----:|------|
| 1. Duplicates | 2 | find_duplicates , |
| 2. Chunking | 3 | chunk_size/overlap , |
| 3. Embedding + Search | 3 | embed , search , |
| 4. | 2 | , parameter |
| **** | **10** | |

---
## ✅ Check answers

Run cell ด้านล่างเพื่อสร้าง **Verification Code** สำหรับส่งงาน

In [ ]:
# ===== cell =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_day1_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 -: {STUDENT_NAME}')
print(f'🎓 : {STUDENT_ID}')
print(f'📱 : {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 : _________________ ()')
print('=' * 50)
print()
print('📋 Checklist :')
print(' [ ] ')
print(' [ ] 1: duplicate + MD5 hash')
print(' [ ] 2: chunks + chunk 1')
print(' [ ] 3: chunk / + ')
print(' [ ] 4: Top-3 scores + ')
print(' [ ] cell run ')